In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

In [2]:
df = pd.read_csv(r"C:\Users\vchan\OneDrive\Desktop\mumbai_local\data\processed\stations_clean.csv")

df.head()

,station,station_code,line,platforms,tracks,year_of_opening,nearby_attractions,distance_km,travel_time_min
0,Churchgate,CCG,Western,4.0,4.0,1867.0,"Marine Drive,Nariman Point,Flora Fountain",0,0
1,Marine Lines,MEL,Western,4.0,4.0,1867.0,"Marine Drive,Girgaum Chowpatty Beach,Taraporew...",1,3
2,Charni Road,CYR,Western,4.0,4.0,1867.0,"Girgaum Chowpatty Beach,Taraporewala Aquarium,...",1,2
3,Grant Road,GTR,Western,4.0,4.0,NaN,"Taraporewala Aquarium,Girgaum Chowpatty Beach,...",1,3
4,Mumbai Central,MMCT,Western,9.0,9.0,1930.0,"Haji Ali Dargah,Mahalakshmi Temple,Nehru Plane...",1,2


In [3]:
coords = {
    "Churchgate": (18.9355, 72.8259),
    "Marine Lines": (18.9431, 72.8237),
    "Charni Road": (18.9511, 72.8196),
    "Grant Road": (18.9611, 72.8164),
    "Mumbai Central": (18.9691, 72.8202),
    "Mahalakshmi": (18.9796, 72.8197),
    "Lower Parel": (18.9930, 72.8282),
    "Prabhadevi": (18.9992, 72.8317),
    "Dadar": (19.0178, 72.8478),
    "Matunga Road": (19.0268, 72.8456),
    "Mahim Jn": (19.0393, 72.8446),
    "Bandra": (19.0543, 72.8394),
    "Khar Road": (19.0655, 72.8363),
    "Santacruz": (19.0814, 72.8389),
    "Vile Parle": (19.0989, 72.8497),
    "Andheri": (19.1194, 72.8468),
    "Jogeshwari": (19.1341, 72.8499),
    "Ram Mandir": (19.1450, 72.8510),
    "Goregaon": (19.1566, 72.8485),
    "Malad": (19.1862, 72.8484),
    "Kandivali": (19.2057, 72.8557),
    "Borivali": (19.2307, 72.8566),
    "Dahisar": (19.2538, 72.8596),
    "Mira Road": (19.2841, 72.8699),
    "Bhayander": (19.3044, 72.8638),
    "Naigaon": (19.3629, 72.8476),
    "Vasai Road": (19.3723, 72.8372),
    "Nalla Sopara": (19.4168, 72.8290),
    "Virar": (19.4588, 72.8112),
    "Vaitarna": (19.5100, 72.7928),
    "Saphale": (19.5700, 72.7750),
    "Kelve Road": (19.6100, 72.7600),
    "Palghar": (19.6958, 72.7653),
    "Umroli Road": (19.7400, 72.7500),
    "Boisar": (19.8000, 72.7400),
    "Vangaon": (19.8700, 72.7300),
    "Dahanu Road": (19.9700, 72.7100),
    "CSMT": (18.9398, 72.8355),
    "Masjid": (18.9467, 72.8373),
    "Sandhurst Road": (18.9550, 72.8379),
    "Byculla": (18.9645, 72.8352),
    "Chinchpokli": (18.9711, 72.8348),
    "Currey Road": (18.9784, 72.8363),
    "Parel": (18.9898, 72.8408),
    "Dadar Central": (19.0196, 72.8445),
    "Matunga": (19.0288, 72.8533),
    "Sion": (19.0392, 72.8617),
    "Kurla": (19.0631, 72.8820),
    "Vidyavihar": (19.0772, 72.9063),
    "Ghatkopar": (19.0863, 72.9069),
    "Vikhroli": (19.1063, 72.9248),
    "Kanjur Marg": (19.1161, 72.9381),
    "Bhandup": (19.1371, 72.9449),
    "Nahur": (19.1511, 72.9437),
    "Mulund": (19.1731, 72.9577),
    "Thane": (19.1877, 72.9744),
    "Kalwa": (19.2011, 73.0007),
    "Mumbra": (19.1869, 73.0226),
    "Diva Jn": (19.1777, 73.0543),
    "Kopar": (19.1690, 73.0800),
    "Dombivli": (19.2153, 73.0865),
    "Thakurli": (19.2230, 73.0933),
    "Kalyan": (19.2438, 73.1303),
    "Shahad": (19.2700, 73.1600),
    "Ambivli": (19.2900, 73.1800),
    "Titvala": (19.3100, 73.2000),
    "Khadavli": (19.3400, 73.2200),
    "Vasind": (19.3700, 73.2500),
    "Asangaon": (19.4700, 73.2800),
    "Atgaon": (19.5300, 73.3000),
    "Khardi": (19.5800, 73.3200),
    "Umbermali": (19.6200, 73.3500),
    "Kasara": (19.6800, 73.4700),
    "Dockyard Road": (18.9531, 72.8434),
    "Reay Road": (18.9598, 72.8510),
    "Cotton Green": (18.9659, 72.8559),
    "Sewri": (18.9762, 72.8624),
    "Wadala Road": (18.9939, 72.8635),
    "GTB Nagar": (19.0149, 72.8617),
    "Chunabhatti": (19.0311, 72.8705),
    "Kurla Harbour": (19.0631, 72.8820),
    "Tilak Nagar": (19.0732, 72.8978),
    "Chembur": (19.0638, 72.8996),
    "Govandi": (19.0543, 72.9135),
    "Mankhurd": (19.0434, 72.9322),
    "Vashi": (19.0764, 73.0091),
    "Sanpada": (19.0657, 73.0097),
    "Juinagar": (19.0529, 73.0171),
    "Nerul": (19.0285, 73.0181),
    "Seawoods Darave": (19.0120, 73.0189),
    "Belapur CBD": (19.0201, 73.0434),
    "Kharghar": (19.0460, 73.0696),
    "Mansarovar": (19.0582, 73.0831),
    "Khandeshwar": (19.0398, 73.0987),
    "Panvel": (18.9938, 73.1086),
}

df['lat'] = df['station'].map(lambda x: coords.get(x, (None, None))[0])
df['lng'] = df['station'].map(lambda x: coords.get(x, (None, None))[1])

print(f"Stations with coords: {df['lat'].notna().sum()} / {len(df)}")
df[['station', 'lat', 'lng']].head(10)

Stations with coords: 114 / 139


,station,lat,lng
0,Churchgate,18.9355,72.8259
1,Marine Lines,18.9431,72.8237
2,Charni Road,18.9511,72.8196
3,Grant Road,18.9611,72.8164
4,Mumbai Central,18.9691,72.8202
5,Mahalakshmi,18.9796,72.8197
6,Lower Parel,18.9930,72.8282
7,Prabhadevi,18.9992,72.8317
8,Dadar,19.0178,72.8478
9,Matunga Road,19.0268,72.8456


In [4]:
missing = df[df['lat'].isna()]['station'].tolist()
print(missing)

['Vidhyavihar', 'Kalva', 'Vithalwadi', 'Ulhas Nagar', 'Ambernath', 'Badalpur', 'Vangani', 'Shelu', 'Neral', 'Bhivpuri Road', 'Karjat', 'Palasdhari', 'Kelavli', 'Dolavli', 'Lowjee', 'Khopoli', 'Vidhyavihar', 'Kalva', 'Titwala', 'Atagaon', 'Thansit', 'Dockyarad Road', 'Vadala Road', 'Tilaknagar', 'Seawood Darave']
